# 03. 온통청년 정책 텍스트마이닝: 키워드, TF-IDF, 토픽모델링, 시각화

이 노트북은 전처리된 정책 텍스트를 사용하여 다음 분석을 수행합니다.

1. 지역별/분류별 정책 수 시각화
2. 전체·지역별·분류별 키워드 빈도 분석
3. TF-IDF 기반 지역별 핵심어 추출
4. LDA 토픽모델링
5. 토픽별 키워드와 지역별 토픽 분포 시각화

## 형태소 분석기 관련 안내

마감이 임박한 학부 프로젝트 상황을 고려해, 별도 Java/KoNLPy 설치 없이 실행 가능한  
`정규표현식 기반 한국어 토큰화`를 기본 방식으로 사용합니다.

더 정교한 분석이 필요하면 추후 `konlpy`, `kiwipiepy`, `mecab` 등을 적용할 수 있습니다.

In [ ]:
# 필요 라이브러리
# scikit-learn이 설치되어 있지 않다면 아래 주석을 해제하고 한 번 실행하세요.
# %pip install scikit-learn

from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

PREPROCESSED_FILE = OUTPUT_DIR / "policy_preprocessed.csv"
RAW_POLICY_FILE = DATA_DIR / "youth_policies_categorized.csv"

def read_csv_auto(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

if PREPROCESSED_FILE.exists():
    df = read_csv_auto(PREPROCESSED_FILE)
    print("전처리 파일 로드:", PREPROCESSED_FILE)
elif RAW_POLICY_FILE.exists():
    df = read_csv_auto(RAW_POLICY_FILE)
    print("전처리 파일이 없어 원본 파일을 로드했습니다. 가능하면 01번 노트북을 먼저 실행하세요.")
else:
    raise FileNotFoundError("policy_preprocessed.csv 또는 youth_policies_categorized.csv를 찾을 수 없습니다.")

print("데이터 크기:", df.shape)
display(df.head())

In [ ]:
# 한글 폰트 설정
plt.rcParams["axes.unicode_minus"] = False

def try_set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    available = {f.name for f in plt.matplotlib.font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            plt.rcParams["font.family"] = font
            return font
    return None

selected_font = try_set_korean_font()
print("사용 폰트:", selected_font or "기본 폰트")

In [ ]:
# 분석 텍스트/분류 컬럼 보정
if "분석텍스트" not in df.columns:
    text_cols = [c for c in ["정책명", "정책키워드", "정책설명", "정책지원내용", "지원대상", "신청방법"] if c in df.columns]
    df["분석텍스트"] = df[text_cols].fillna("").agg(" ".join, axis=1)

if "대표분류_정리" not in df.columns:
    df["대표분류_정리"] = df.get("대표분류", df.get("자동분류", "기타")).fillna("기타")

df["분석텍스트"] = df["분석텍스트"].fillna("").astype(str)
df = df[df["분석텍스트"].str.len() > 0].copy()

print("분석 대상 정책 수:", len(df))
display(df[["지역", "정책명", "대표분류_정리", "분석텍스트"]].head())

In [ ]:
# 기본 시각화 1: 지역별 정책 수
region_counts = df["지역"].value_counts().sort_values()

plt.figure(figsize=(10, 6))
plt.barh(region_counts.index, region_counts.values)
plt.xlabel("정책 수")
plt.ylabel("지역")
plt.title("지역별 온통청년 정책 수")
plt.tight_layout()

fig_path = FIG_DIR / "policy_count_by_region.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("그림 저장:", fig_path)

In [ ]:
# 기본 시각화 2: 지역×분류 정책 수 피벗
category_order = ["일자리", "직무교육", "주거지원", "창업지원", "복지", "참여 프로그램", "기타"]

region_category = (
    pd.pivot_table(
        df,
        index="지역",
        columns="대표분류_정리",
        values="정책명",
        aggfunc="count",
        fill_value=0
    )
    .reindex(columns=category_order, fill_value=0)
)

display(region_category)

region_category.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.xlabel("지역")
plt.ylabel("정책 수")
plt.title("지역별 정책 분야 구성")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig_path = FIG_DIR / "policy_category_stacked_bar.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("그림 저장:", fig_path)

## 토큰화 및 불용어 기준

정책 텍스트에서 다음 기준으로 단어를 추출합니다.

- 한글/영문 2글자 이상 단어만 추출
- 조사·접속어·분석에 불필요한 일반어 제거
- `청년`, `지원`, `정책`처럼 너무 흔하지만 프로젝트 해석에 따라 필요한 단어는 일부 유지 가능
- 너무 일반적인 단어는 아래 `STOPWORDS`에 추가하면 됩니다.

In [ ]:
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z]{2,}")

STOPWORDS = {
    "그리고", "그러나", "또는", "대한", "대해", "위한", "통해", "관련", "경우", "기타",
    "가능", "지원", "사업", "정책", "청년", "대상", "신청", "제공", "운영", "추진",
    "지역", "해당", "이상", "이하", "미만", "초과", "개월", "년도", "지원자",
    "선정", "모집", "안내", "확인", "홈페이지", "온라인", "오프라인",
    "프로그램", "서비스", "내용", "방법", "기관", "기간", "참여", "활동",
    "대한민국", "한국", "정부", "지자체"
}

def tokenize_ko(text):
    text = str(text)
    text = re.sub(r"\b\d{7}\b", " ", text)
    tokens = TOKEN_PATTERN.findall(text)
    tokens = [t.lower() for t in tokens]
    tokens = [t for t in tokens if len(t) >= 2 and t not in STOPWORDS]
    return tokens

sample_text = df["분석텍스트"].iloc[0]
print("원문 예시:", sample_text[:200])
print("토큰 예시:", tokenize_ko(sample_text)[:30])

In [ ]:
# 전체 키워드 빈도
all_tokens = []
for text in df["분석텍스트"]:
    all_tokens.extend(tokenize_ko(text))

overall_freq = pd.DataFrame(Counter(all_tokens).most_common(100), columns=["키워드", "빈도"])
display(overall_freq.head(30))

overall_freq.to_csv(OUTPUT_DIR / "token_frequency_overall.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 전체 상위 키워드 시각화
top_n = 25
plot_df = overall_freq.head(top_n).sort_values("빈도", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["키워드"], plot_df["빈도"])
plt.xlabel("빈도")
plt.ylabel("키워드")
plt.title(f"전체 정책 텍스트 상위 키워드 Top {top_n}")
plt.tight_layout()

fig_path = FIG_DIR / "top_keywords_overall.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("그림 저장:", fig_path)

In [ ]:
# 지역별 상위 키워드
region_keyword_rows = []

for region, group in df.groupby("지역"):
    tokens = []
    for text in group["분석텍스트"]:
        tokens.extend(tokenize_ko(text))
    for rank, (word, cnt) in enumerate(Counter(tokens).most_common(20), start=1):
        region_keyword_rows.append({
            "지역": region,
            "순위": rank,
            "키워드": word,
            "빈도": cnt
        })

region_keywords = pd.DataFrame(region_keyword_rows)
display(region_keywords.head(40))

region_keywords.to_csv(OUTPUT_DIR / "top_keywords_by_region.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 분류별 상위 키워드
category_keyword_rows = []

for category, group in df.groupby("대표분류_정리"):
    tokens = []
    for text in group["분석텍스트"]:
        tokens.extend(tokenize_ko(text))
    for rank, (word, cnt) in enumerate(Counter(tokens).most_common(20), start=1):
        category_keyword_rows.append({
            "대표분류": category,
            "순위": rank,
            "키워드": word,
            "빈도": cnt
        })

category_keywords = pd.DataFrame(category_keyword_rows)
display(category_keywords.head(40))

category_keywords.to_csv(OUTPUT_DIR / "top_keywords_by_category.csv", index=False, encoding="utf-8-sig")

## TF-IDF 분석

TF-IDF는 단순히 자주 등장하는 단어가 아니라, 특정 지역에서 상대적으로 두드러지는 단어를 찾는 데 유용합니다.

- `min_df=2`: 최소 2개 정책 이상에서 등장한 단어만 사용
- `max_df=0.85`: 전체 문서의 85% 이상에서 등장하는 너무 흔한 단어 제거
- `max_features=3000`: 상위 3,000개 단어까지만 사용

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize_ko,
    token_pattern=None,
    min_df=2,
    max_df=0.85,
    max_features=3000
)

X_tfidf = tfidf_vectorizer.fit_transform(df["분석텍스트"])
terms = np.array(tfidf_vectorizer.get_feature_names_out())

print("TF-IDF 행렬 크기:", X_tfidf.shape)

In [ ]:
# 지역별 평균 TF-IDF 상위어 추출
tfidf_region_rows = []

for region in sorted(df["지역"].dropna().unique()):
    idx = df.index[df["지역"] == region].tolist()
    # df.index가 연속이 아닐 수 있어 위치 기준으로 변환
    positions = [df.index.get_loc(i) for i in idx]
    mean_scores = np.asarray(X_tfidf[positions].mean(axis=0)).ravel()
    top_idx = mean_scores.argsort()[::-1][:20]

    for rank, term_idx in enumerate(top_idx, start=1):
        tfidf_region_rows.append({
            "지역": region,
            "순위": rank,
            "키워드": terms[term_idx],
            "TFIDF": round(float(mean_scores[term_idx]), 6)
        })

tfidf_by_region = pd.DataFrame(tfidf_region_rows)
display(tfidf_by_region.head(50))

tfidf_by_region.to_csv(OUTPUT_DIR / "top_tfidf_by_region.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 지역별 TF-IDF Top 10 키워드 간단 출력
for region in sorted(tfidf_by_region["지역"].unique()):
    words = tfidf_by_region.query("지역 == @region").head(10)["키워드"].tolist()
    print(f"{region}: {', '.join(words)}")

## LDA 토픽모델링

LDA는 전체 정책 문서를 여러 개의 잠재 토픽으로 나누고, 각 토픽을 대표하는 키워드를 추출하는 기법입니다.

기본 토픽 수는 6개로 설정했습니다.  
발표/보고서에서 해석하기 어렵다면 `N_TOPICS = 5` 또는 `N_TOPICS = 7`로 바꿔 비교해도 됩니다.

In [ ]:
N_TOPICS = 6

count_vectorizer = CountVectorizer(
    tokenizer=tokenize_ko,
    token_pattern=None,
    min_df=2,
    max_df=0.85,
    max_features=3000
)

X_count = count_vectorizer.fit_transform(df["분석텍스트"])
count_terms = np.array(count_vectorizer.get_feature_names_out())

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch",
    max_iter=20
)

topic_matrix = lda.fit_transform(X_count)

print("Count 행렬 크기:", X_count.shape)
print("토픽 분포 행렬 크기:", topic_matrix.shape)

In [ ]:
# 토픽별 상위 키워드
topic_rows = []

for topic_idx, topic_weights in enumerate(lda.components_):
    top_indices = topic_weights.argsort()[::-1][:15]
    for rank, term_idx in enumerate(top_indices, start=1):
        topic_rows.append({
            "토픽번호": topic_idx + 1,
            "순위": rank,
            "키워드": count_terms[term_idx],
            "가중치": round(float(topic_weights[term_idx]), 4)
        })

topic_keywords = pd.DataFrame(topic_rows)
display(topic_keywords)

topic_keywords.to_csv(OUTPUT_DIR / "topic_keywords.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 각 정책의 대표 토픽 부여
topic_cols = [f"토픽{i+1}_확률" for i in range(N_TOPICS)]
topic_df = pd.DataFrame(topic_matrix, columns=topic_cols)

df_topics = pd.concat([df.reset_index(drop=True), topic_df], axis=1)
df_topics["대표토픽"] = topic_matrix.argmax(axis=1) + 1
df_topics["대표토픽확률"] = topic_matrix.max(axis=1).round(4)

topic_assignment_cols = ["지역", "정책명", "대표분류_정리", "대표토픽", "대표토픽확률"] + topic_cols
display(df_topics[topic_assignment_cols].head())

df_topics.to_csv(OUTPUT_DIR / "policy_topic_assignment.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 지역별 토픽 분포
topic_by_region = (
    df_topics.groupby(["지역", "대표토픽"])
    .size()
    .reset_index(name="정책수")
)

topic_pivot = (
    topic_by_region
    .pivot(index="지역", columns="대표토픽", values="정책수")
    .fillna(0)
    .astype(int)
)

topic_pivot_ratio = topic_pivot.div(topic_pivot.sum(axis=1), axis=0)

display(topic_pivot)
display(topic_pivot_ratio.round(3))

topic_pivot.to_csv(OUTPUT_DIR / "topic_distribution_by_region_count.csv", encoding="utf-8-sig")
topic_pivot_ratio.to_csv(OUTPUT_DIR / "topic_distribution_by_region_ratio.csv", encoding="utf-8-sig")

In [ ]:
# 지역별 토픽 분포 시각화
topic_pivot_ratio.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.xlabel("지역")
plt.ylabel("비율")
plt.title("지역별 정책 토픽 분포")
plt.xticks(rotation=45, ha="right")
plt.legend(title="대표토픽", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

fig_path = FIG_DIR / "topic_distribution_by_region.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("그림 저장:", fig_path)

In [ ]:
# 토픽 해석 보조표 생성
topic_label_suggestions = []

for topic_num in range(1, N_TOPICS + 1):
    words = topic_keywords.query("토픽번호 == @topic_num").head(10)["키워드"].tolist()
    topic_label_suggestions.append({
        "토픽번호": topic_num,
        "상위키워드": ", ".join(words),
        "해석라벨_수동입력": ""
    })

topic_labels = pd.DataFrame(topic_label_suggestions)
display(topic_labels)

topic_labels.to_csv(OUTPUT_DIR / "topic_label_suggestions.csv", index=False, encoding="utf-8-sig")

## 보고서에 쓸 수 있는 분석 흐름 예시

본 연구에서는 온통청년 정책명, 정책키워드, 정책설명, 정책지원내용, 지원대상, 신청방법 등을 결합하여 정책별 분석 텍스트를 구성하였다.  
이후 정규표현식 기반 한국어 토큰화를 적용하고 불용어를 제거한 뒤, 빈도 분석을 통해 전체 및 지역별 주요 키워드를 확인하였다.  
또한 TF-IDF를 활용하여 각 지역에서 상대적으로 두드러지는 정책 키워드를 도출하였으며, LDA 토픽모델링을 통해 정책 텍스트가 어떠한 주제군으로 구성되는지 분석하였다.  
마지막으로 지역별 토픽 분포를 비교하여 지역별 정책 공급 방향의 차이를 해석하였다.